In [13]:
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [15]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    
    if system:
        params["system"] = system

    message= client.messages.create(**params)
    return message.content[0].text

In [16]:
import json

In [17]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages= [] 
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [18]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TO DO - Grading 
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [22]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [23]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [24]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# Solution: Parse AWS S3 Bucket Name from URI\n\n```python\ndef parse_s3_bucket_name(s3_uri):\n    \"\"\"\n    Parses an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \"\"\"\n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"Invalid S3 URI: must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    bucket_name = uri_without_prefix.split('/')[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Invalid S3 URI: bucket name is empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Basic test\n    print(parse_s3_bucket_name('s3://my-bucket/key/path'))  # Output: my-bucket\n    \n    # With trailin